# 2. Route a question to the right step

A workflow can choose its next step by returning a different event type. Here an LLM classifies a question, and the workflow routes it to either a documentation response or a general response.

This is a routing exercise. The documentation branch does not search yet; notebook 3 adds evidence from GoodMem.

In [ ]:
from typing import Literal
from pydantic import BaseModel
from llama_index.core.prompts import PromptTemplate
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step
from goodmem_rag.config import chat_model
from goodmem_rag.agents import chat

model = chat_model()

class Route(BaseModel):
    category: Literal["documentation", "general"]

class DocumentationQuestion(Event):
    question: str

class GeneralQuestion(Event):
    question: str

class Router(Workflow):
    @step
    async def classify(self, ev: StartEvent) -> DocumentationQuestion | GeneralQuestion:
        route = await model.astructured_predict(Route, PromptTemplate(
            "Classify software-framework questions as documentation; everything else as general. "
            "Question: {question}"), question=ev.question)
        event = DocumentationQuestion if route.category == "documentation" else GeneralQuestion
        return event(question=ev.question)

    @step
    async def documentation(self, ev: DocumentationQuestion) -> StopEvent:
        return StopEvent(result={"route": "documentation", "answer":
            "This question needs documentation evidence. Notebook 3 connects this branch to GoodMem."})

    @step
    async def general(self, ev: GeneralQuestion) -> StopEvent:
        answer = await chat(model, "Answer in one sentence.", ev.question)
        return StopEvent(result={"route": "general", "answer": answer})

router = Router(timeout=120)
print(await router.run(question="How does a LangGraph checkpointer work?"))
print(await router.run(question="What is the capital of France?"))

The event type determines which function runs. Compare this with a graph API that names a conditional edge: both express routing, but LlamaIndex makes the next step's input type part of the connection.